In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

In [ ]:
import os
import cv2
import numpy as np
import yaml
import shutil
from tqdm import tqdm
from ultralytics import YOLO

In [ ]:
SRC_IMG_TRAIN = "Images/img_dir/train"
SRC_IMG_VAL = "Images/img_dir/test"
SRC_ANN_TRAIN = "Images/ann_dir/train"
SRC_ANN_VAL = "Images/ann_dir/test"
CATEGORY_FILE = "category_id.txt"

In [ ]:
YOLO_ROOT = "yolo_dataset"
dirs = [
    f"{YOLO_ROOT}/images/train", f"{YOLO_ROOT}/images/val",
    f"{YOLO_ROOT}/labels/train", f"{YOLO_ROOT}/labels/val"
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

In [ ]:
classes = []
with open(CATEGORY_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            parts = line.strip().split()
            name = parts[-1] 
            classes.append(name)
print(f"Загружено классов: {len(classes)}")

In [ ]:
def convert_and_copy(img_src, ann_src, img_dest, label_dest):
    print(f"Обработка: {img_src}...")
    
    mask_files = [f for f in os.listdir(ann_src) if f.endswith('.png')]
    
    for mask_name in tqdm(mask_files):
        img_name = mask_name.replace('.png', '.jpg')
        src_img_path = os.path.join(img_src, img_name)
        
        
        if not os.path.exists(src_img_path):
            img_name = mask_name.replace('.png', '.png')
            src_img_path = os.path.join(img_src, img_name)

        if os.path.exists(src_img_path):
            shutil.copy(src_img_path, os.path.join(img_dest, img_name))
        else:
            continue

        mask_path = os.path.join(ann_src, mask_name)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None: continue
        
        h, w = mask.shape[:2]
        cls_ids = np.unique(mask)
        
        txt_name = mask_name.rsplit('.', 1)[0] + ".txt"
        with open(os.path.join(label_dest, txt_name), 'w') as f:
            for cls_id in cls_ids:
                if cls_id == 0: continue
                
                binary_mask = np.where(mask == cls_id, 255, 0).astype(np.uint8)
                contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                for contour in contours:
                    if len(contour) < 3: continue 
                    
                    polygon = contour.reshape(-1, 2) / [w, h]
                    poly_str = " ".join([f"{c[0]:.4f} {c[1]:.4f}" for c in polygon])
                    
                    f.write(f"{int(cls_id)-1} {poly_str}\n")

In [ ]:
convert_and_copy(SRC_IMG_TRAIN, SRC_ANN_TRAIN, f"{YOLO_ROOT}/images/train", f"{YOLO_ROOT}/labels/train")
convert_and_copy(SRC_IMG_VAL, SRC_ANN_VAL, f"{YOLO_ROOT}/images/val", f"{YOLO_ROOT}/labels/val")

In [ ]:
data_yaml = {
    'path': os.path.abspath(YOLO_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'names': {i: name for i, name in enumerate(classes)}
}

with open('food103.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print("Данные готовы! Файл food103.yaml создан.")

In [ ]:
model = YOLO("yolo26s-seg.pt")

model.train(
    data="food103.yaml",
    epochs=100,
    imgsz=640,
    batch=32, 
    device=0,   
    project="food_project",
    name="yolo26_foodseg_run",
    verbose=True,
    plots=True
)